# 7.4.2 学習済み単語ベクトルの今の書き方

`7-4_vectorize.ipynb` は、日本語の word2vec と fastText を `torchtext.data.Field` と `torchtext.vocab.Vectors` で語彙に載せる。このファイルは、同じデータと前処理を、今の PyTorch の `Dataset` と辞書で書き直したものである。

文章の読み込みは `7-2-2_Dataset_DataLoader.ipynb` と同じ役割である。7-4 ではそのあと、学習済みベクトルを `TEXT.vocab.vectors` に並べている。ここでは、単語から ID への辞書 `stoi` と同じ順の行列にベクトルを入れる。

7-5 以降は、まだ 7-4 の `TEXT.vocab` を前提にしている。このファイルは 7-4 の読み替え用で、後ろの章はそのままにする。

# 7.4.2 学習目標

1.	学習済みの日本語 word2vec で単語をベクトルにする処理を、`Dataset` と辞書で実装できる
2.	学習済みの日本語 fastText で同じ処理を実装できる

# 事前準備

- 文章は 7-4 と同じ `data/text_train.tsv`、`data/text_val.tsv`、`data/text_test.tsv`
- word2vec は `data/entity_vector/entity_vector.model.bin`
- fastText は `data/vector_neologd/model.vec`
- 単語分割は 7-4 と同じ MeCab+NEologd

# 1. 文書を読み込んで、分かち書き、データセット作成まで

7-4 のこの節は、7-2 の `Field` と `TabularDataset` と同じ書き方である。分かち書きの関数は 7-4 と同じ MeCab+NEologd を使う。データセットの形は 7-2-2 の `TextDataset` に合わせる。

In [2]:
# 単語分割にはMecab＋NEologdを使用
import MeCab

m_t = MeCab.Tagger('-Owakati -d /usr/lib/mecab/dic/mecab-ipadic-neologd')


def tokenizer_mecab(text):
    text = m_t.parse(text)  # これでスペースで単語が区切られる
    ret = text.strip().split()  # スペース部分で区切ったリストに変換
    return ret

In [3]:
# 前処理として正規化をする関数を定義
import re


def preprocessing_text(text):
    # 改行、半角スペース、全角スペースを削除
    text = re.sub('\r', '', text)
    text = re.sub('\n', '', text)
    text = re.sub('　', '', text)
    text = re.sub(' ', '', text)

    # 数字文字の一律「0」化
    text = re.sub(r'[0-9 ０-９]', '0', text)  # 数字

    return text


def tokenizer_with_preprocessing(text):
    text = preprocessing_text(text)  # 前処理の正規化
    ret = tokenizer_mecab(text)  # Mecabの単語分割
    return ret

In [4]:
from collections import Counter

from torch.utils.data import Dataset


class TextDataset(Dataset):
    """tsv の1行を (単語リスト, ラベル) として持つ。7-4 の TabularDataset に相当する。"""

    def __init__(self, path, tokenizer):
        self.samples = []
        with open(path, encoding="utf-8") as f:
            for line in f:
                text, label = line.rstrip("\n").split("\t")
                # Field の lower=True と同じく、分割の前に小文字化する
                tokens = tokenizer(text.lower())
                self.samples.append((tokens, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        return self.samples[index]


def build_vocab(dataset, min_freq=1):
    """7-4 の TEXT.build_vocab のうち、単語と ID の対応を作る部分。"""
    freqs = Counter()
    for tokens, _ in dataset:
        freqs.update(tokens)

    words = [word for word, freq in freqs.items() if freq >= min_freq]
    words.sort(key=lambda word: (-freqs[word], word))

    itos = ["<unk>", "<pad>"] + words
    stoi = {word: index for index, word in enumerate(itos)}
    return stoi, itos


train_ds = TextDataset("./data/text_train.tsv", tokenizer_with_preprocessing)
val_ds = TextDataset("./data/text_val.tsv", tokenizer_with_preprocessing)
test_ds = TextDataset("./data/text_test.tsv", tokenizer_with_preprocessing)

stoi, itos = build_vocab(train_ds, min_freq=1)
print("単語数:", len(stoi))
print("1つ目の訓練データ", {"Text": train_ds[0][0], "Label": train_ds[0][1]})
print("2つ目の訓練データ", {"Text": train_ds[1][0], "Label": train_ds[1][1]})

単語数: 49
1つ目の訓練データ {'Text': ['王', 'と', '王子', 'と', '女王', 'と', '姫', 'と', '男性', 'と', '女性', 'が', 'い', 'まし', 'た', '。'], 'Label': '0'}
2つ目の訓練データ {'Text': ['機械学習', 'が', '好き', 'です', '。'], 'Label': '1'}


# 2. 単語のベクトル化

7-4 は `torchtext.vocab.Vectors` で学習済みファイルを読み、`TEXT.build_vocab(..., vectors=...)` で `TEXT.vocab.vectors` を作っていた。

ここでは gensim の `KeyedVectors` で同じファイルを読み、`stoi` の順番にベクトルを並べた行列を作る。`<unk>` と `<pad>`、学習済みファイルに無い単語は、7-4 と同じくゼロベクトルにする。

word2vec のバイナリは、7-4 では torchtext が読めないのでテキスト形式へ保存し直していた。gensim はそのバイナリを直接読めるので、保存し直しはしない。

In [5]:
import torch

def build_vector_table(keyed_vectors, stoi):
    """7-4 の TEXT.vocab.vectors に相当する行列を作る。"""
    table = torch.zeros(len(stoi), keyed_vectors.vector_size)
    for word, index in stoi.items():
        if word in keyed_vectors:
            table[index] = torch.tensor(keyed_vectors[word].copy())
    return table

## 2.1 word2vec

7-4 と同じく、東北大学 乾・岡崎研究室の日本語 Wikipedia エンティティベクトルを使う。

7-4 の対応するコードは次である。

```python
model = KeyedVectors.load_word2vec_format(
    './data/entity_vector/entity_vector.model.bin', binary=True)
model.save_word2vec_format('./data/japanese_word2vec_vectors.vec')

japanese_word2vec_vectors = Vectors(name='./data/japanese_word2vec_vectors.vec')
TEXT.build_vocab(train_ds, vectors=japanese_word2vec_vectors, min_freq=1)
```

In [6]:
from gensim.models import KeyedVectors

word2vec = KeyedVectors.load_word2vec_format(
    "./data/entity_vector/entity_vector.model.bin", binary=True)

print("1単語を表現する次元数:", word2vec.vector_size)
print("学習済みの単語数:", len(word2vec.vocab))

word2vec_vectors = build_vector_table(word2vec, stoi)
print(word2vec_vectors.shape)  # 7-4 では torch.Size([49, 200])
word2vec_vectors

1単語を表現する次元数: 200
学習済みの単語数: 1015474
torch.Size([49, 200])


tensor([[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 2.6023, -2.6357, -2.5822,  ...,  0.6953, -1.4977,  1.4752],
        ...,
        [-2.8353,  2.5609, -0.5348,  ...,  0.4602,  1.4669, -2.1255],
        [-1.5885,  0.1614, -0.6029,  ..., -1.7545, -1.2462,  2.3034],
        [-0.0448, -0.1304,  0.0329,  ...,  0.0825, -0.1386,  0.0417]])

In [7]:
# 単語の順番。7-4 の TEXT.vocab.stoi に相当する
stoi

{'<unk>': 0,
 '<pad>': 1,
 'と': 2,
 '。': 3,
 'な': 4,
 'の': 5,
 '文章': 6,
 '、': 7,
 'が': 8,
 'し': 9,
 'を': 10,
 'いる': 11,
 'か': 12,
 'て': 13,
 'ます': 14,
 '分類': 15,
 '本章': 16,
 '評価': 17,
 '0': 18,
 'い': 19,
 'から': 20,
 'する': 21,
 'その': 22,
 'た': 23,
 'で': 24,
 'です': 25,
 'に': 26,
 'に対して': 27,
 'は': 28,
 'まし': 29,
 'クラス': 30,
 'ネガティブ': 31,
 'ポジティブ': 32,
 'モデル': 33,
 'レビュー': 34,
 '値': 35,
 '取り組み': 36,
 '商品': 37,
 '女性': 38,
 '女王': 39,
 '好き': 40,
 '姫': 41,
 '構築': 42,
 '機械学習': 43,
 '王': 44,
 '王子': 45,
 '男性': 46,
 '短い': 47,
 '自然言語処理': 48}

7-4 は `姫 - 女性 + 男性` を、ベクトルの番号 41、38、46 で計算していた。番号は `stoi` の並びなので、ここでは単語から番号を取る。

In [8]:
import torch.nn.functional as F

# 姫 - 女性 + 男性
tensor_calc = word2vec_vectors[stoi["姫"]] - word2vec_vectors[stoi["女性"]] + word2vec_vectors[stoi["男性"]]

# コサイン類似度を計算
# dim=0 は0次元目で計算してくださいという指定
print("女王", F.cosine_similarity(tensor_calc, word2vec_vectors[stoi["女王"]], dim=0))
print("王", F.cosine_similarity(tensor_calc, word2vec_vectors[stoi["王"]], dim=0))
print("王子", F.cosine_similarity(tensor_calc, word2vec_vectors[stoi["王子"]], dim=0))
print("機械学習", F.cosine_similarity(tensor_calc, word2vec_vectors[stoi["機械学習"]], dim=0))

del word2vec

女王 tensor(0.3840)
王 tensor(0.3669)
王子 tensor(0.5489)
機械学習 tensor(-0.1404)


姫 - 女性 + 男性 を計算すると、7-4 と同じく王子がもっとも近い結果になる。

## 2.2 fastText

7-4 と同じく、`data/vector_neologd/model.vec` を使う。このファイルは最初からテキスト形式なので、7-4 では `Vectors` でそのまま読んでいた。

```python
japanese_fasttext_vectors = Vectors(name='./data/vector_neologd/model.vec')
TEXT.build_vocab(train_ds, vectors=japanese_fasttext_vectors, min_freq=1)
```

読み込み方は word2vec と同じで、`binary=False` にする。

In [9]:
fasttext = KeyedVectors.load_word2vec_format(
    "./data/vector_neologd/model.vec", binary=False)

print("1単語を表現する次元数:", fasttext.vector_size)
print("学習済みの単語数:", len(fasttext.vocab))

fasttext_vectors = build_vector_table(fasttext, stoi)
print(fasttext_vectors.shape)  # 7-4 では torch.Size([49, 300])
fasttext_vectors

1単語を表現する次元数: 300
学習済みの単語数: 351122
torch.Size([49, 300])


tensor([[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [-0.0182, -0.0894, -0.1011,  ..., -0.0410, -0.0735,  0.0045],
        ...,
        [ 0.0161, -0.0172, -0.2945,  ..., -0.0038,  0.1588, -0.0076],
        [-0.2099,  0.1631, -0.0182,  ..., -0.4823,  0.0375,  0.0088],
        [-0.2762,  0.2986,  0.0511,  ..., -0.1052, -0.1779, -0.1957]])

In [10]:
# 姫 - 女性 + 男性
tensor_calc = fasttext_vectors[stoi["姫"]] - fasttext_vectors[stoi["女性"]] + fasttext_vectors[stoi["男性"]]

print("女王", F.cosine_similarity(tensor_calc, fasttext_vectors[stoi["女王"]], dim=0))
print("王", F.cosine_similarity(tensor_calc, fasttext_vectors[stoi["王"]], dim=0))
print("王子", F.cosine_similarity(tensor_calc, fasttext_vectors[stoi["王子"]], dim=0))
print("機械学習", F.cosine_similarity(tensor_calc, fasttext_vectors[stoi["機械学習"]], dim=0))

del fasttext

女王 tensor(0.3650)
王 tensor(0.3461)
王子 tensor(0.5531)
機械学習 tensor(0.0952)


姫 - 女性 + 男性 を計算すると、7-4 と同じく王子がもっとも近い結果になる。

以上